In [ ]:
import pandas as pd
import os
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "olist_dw")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}/{DB_NAME}")

# 2. Folder Path & Mapping of Table Names -> CSV Filenames
data_dir = "C:/Users/HP/Desktop/Projects/olist-Ecommerce-analytics-warehouse/Data"

files_to_tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# 3. Date columns - DATETIME instead of TEXT
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"]
}

# 4. Loop and Load Each Table
for table_name, file_name in files_to_tables.items():
    file_path = os.path.join(data_dir, file_name)
    print(f"Loading '{table_name}' from {file_name}...")
    
    df = pd.read_csv(file_path)
    
    # Convert date columns to true datetime format
    if table_name in date_columns:
        for col in date_columns[table_name]:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            
    # Push to MySQL in chunks of 5,000 rows 
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",  # Overwrites if re-run the script
        index=False,
        chunksize=5000
    )
    print(f"  -> Done! Inserted {len(df):,} rows into `{table_name}`.")

print("\nAll 9 tables successfully loaded into olist_dw!")

Loading 'orders' from olist_orders_dataset.csv...
  -> Done! Inserted 99,441 rows into `orders`.
Loading 'customers' from olist_customers_dataset.csv...
  -> Done! Inserted 99,441 rows into `customers`.
Loading 'order_items' from olist_order_items_dataset.csv...
  -> Done! Inserted 112,650 rows into `order_items`.
Loading 'order_payments' from olist_order_payments_dataset.csv...
  -> Done! Inserted 103,886 rows into `order_payments`.
Loading 'order_reviews' from olist_order_reviews_dataset.csv...
  -> Done! Inserted 99,224 rows into `order_reviews`.
Loading 'products' from olist_products_dataset.csv...
  -> Done! Inserted 32,951 rows into `products`.
Loading 'sellers' from olist_sellers_dataset.csv...
  -> Done! Inserted 3,095 rows into `sellers`.
Loading 'geolocation' from olist_geolocation_dataset.csv...
  -> Done! Inserted 1,000,163 rows into `geolocation`.
Loading 'category_translation' from product_category_name_translation.csv...
  -> Done! Inserted 71 rows into `category_transla